#AI-Powered Business FAQ Chatbot

In [1]:
!pip install sentence-transformers faiss-cpu pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 81.6 MB/s eta 0:00:00


In [3]:
import pandas as pd

df = pd.read_csv("faq_data.csv")
print(df.shape)
df.head()

(100, 4)


,id,category,question,answer
0,1,Ordering,How do I place an order on your website?,"Just browse our grocery categories, add items ..."
1,2,Ordering,Is there a mobile app I can use to order?,"Yes, our app is available on both Android and ..."
2,3,Ordering,Can I order groceries without creating an acco...,You'll need to create a free account first so ...
3,4,Ordering,Is there a minimum order value?,"Yes, orders must be at least Rs. 1,000 in valu..."
4,5,Ordering,What is the smallest amount I can order?,"The minimum order value is Rs. 1,000, excludin..."


In [4]:
#100 questions into embeddings

from sentence_transformers import SentenceTransformer

# This downloads a small, free, pretrained model (only ~80MB)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert every question into an embedding (a list of numbers representing its meaning)
question_embeddings = model.encode(df['question'].tolist())

print(question_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(100, 384)


In [5]:
#search part

import faiss
import numpy as np

# FAISS needs float32 numbers
question_embeddings = np.array(question_embeddings).astype('float32')

# Create a simple search index and add our embeddings to it
dimension = question_embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(question_embeddings)

print("Number of questions in the search index:", index.ntotal)

Number of questions in the search index: 100


In [6]:
#test

def search_faq(user_question, top_k=1):
    # Convert the user's question into an embedding, same way as before
    query_embedding = model.encode([user_question]).astype('float32')

    # Search the index for the closest match(es)
    distances, indices = index.search(query_embedding, top_k)

    for idx in indices[0]:
        print("Matched Question:", df.iloc[idx]['question'])
        print("Answer:", df.iloc[idx]['answer'])
        print("---")

# Try it with a question that ISN'T worded exactly like your CSV
search_faq("When will my groceries arrive?")

Matched Question: When will my order arrive?
Answer: Most orders are delivered within 1-2 days. You'll get an estimated delivery window when you place your order.
---


free LLM - Groq

In [7]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.6 MB/s eta 0:00:00


In [8]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def generate_reply(matched_answer, user_question):
    prompt = f"""A customer asked: "{user_question}"
The stored answer is: "{matched_answer}"

Rephrase this answer in a warm, natural, and polite tone for a customer support chat. Keep it short and clear."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

# Test it
reply = generate_reply(
    "Most orders are delivered within 1-2 days. You'll get an estimated delivery window when you place your order.",
    "When will my groceries arrive?"
)
print(reply)

"You can expect your groceries to arrive within 1-2 days. As soon as you place your order, you'll receive an estimated delivery time frame, so you know exactly when to expect it."


In [9]:
#more test

test_questions = [
    "Can I get a refund if my card was charged wrong?",
    "hey can u guys deliver alcohol on poya day",
    "how many times can i use a coupon code"
]

for q in test_questions:
    matches = search_faq(q)
    print(f"USER ASKED: {q}\n")

Matched Question: Why was my card charged before delivery?
Answer: For card payments, we place a temporary hold for the order value. Only the actual delivered amount is finally charged, and the rest is released automatically.
---
USER ASKED: Can I get a refund if my card was charged wrong?

Matched Question: Can I get my order delivered on a Poya day?
Answer: General groceries can still be delivered, but alcohol and meat products are excluded on Poya days by law.
---
USER ASKED: hey can u guys deliver alcohol on poya day

Matched Question: Can I use a discount code more than once?
Answer: No, each promo code is valid for a single use per customer account.
---
USER ASKED: how many times can i use a coupon code



In [10]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 58.7 MB/s eta 0:00:00


In [11]:
#write chatbot app to a file

%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

st.set_page_config(page_title="Grocery FAQ Bot", page_icon="🛒")
st.title("🛒 FreshMart FAQ Assistant")
st.caption("Ask me anything about ordering, delivery, payments, or returns!")

@st.cache_resource
def load_everything():
    df = pd.read_csv("faq_data.csv")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['question'].tolist()).astype('float32')
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return df, model, index

df, model, index = load_everything()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

def search_faq(user_question, top_k=1):
    query_embedding = model.encode([user_question]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    idx = indices[0][0]
    return df.iloc[idx]['question'], df.iloc[idx]['answer']

def generate_reply(matched_answer, user_question):
    prompt = f"""A customer asked: "{user_question}"
The stored answer is: "{matched_answer}"

Rephrase this answer in a warm, natural, and polite tone for a customer support chat. Keep it short and clear."""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("Type your question here...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    matched_q, matched_a = search_faq(user_input)
    reply = generate_reply(matched_a, user_input)

    st.session_state.messages.append({"role": "assistant", "content": reply})
    with st.chat_message("assistant"):
        st.write(reply)

Writing app.py


In [14]:
#launch a live preview link

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

from pyngrok import ngrok
import threading

# Set your ngrok authtoken
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

def run_app():
    os.system("streamlit run app.py --server.port 8501")

thread = threading.Thread(target=run_app)
thread.start()

import time
time.sleep(5)
public_url = ngrok.connect(8501)
print("Your chatbot is live at:", public_url)

Your chatbot is live at: NgrokTunnel: "https://yonder-doorframe-broadcast.ngrok-free.dev" -> "http://localhost:8501"


In [15]:
from google.colab import files

In [16]:
files.download('app.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
files.download('faq_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>